In [1]:
!nvidia-smi

Sun Aug 23 12:10:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
%%writefile syncexample.cu
#include <iostream>
#include <cuda_runtime.h>

#define N 8

__global__ void reverseArrayKernel(int *d_out, int *d_in){
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx<N) d_out[idx] = d_in[N-idx-1];

}

int main(){
  int h_in[N] = {10,20,30,40,50,60,70,80};
  int h_out[N] = {0};

  int *d_in, *d_out;

  cudaMalloc((void**)&d_in, N*sizeof(int));
  cudaMalloc((void**)&d_out, N*sizeof(int));

  cudaMemcpy(d_in, h_in, N*sizeof(int), cudaMemcpyHostToDevice);

  reverseArrayKernel<<<1,N>>>(d_out, d_in);

  cudaMemcpy(h_out, d_out, N*sizeof(int), cudaMemcpyDeviceToHost);

  std::cout << "Input Array:  ";
  for (int i=0;i<N;i++) std::cout <<h_in[i] << " ";
  std::cout << "\nReversed:    ";
  for (int i=0;i<N;i++) std::cout <<h_out[i] << " ";
  std::cout << std::endl;

  cudaFree(d_in);
  cudaFree(d_out);

  return 0;

}


Overwriting syncexample.cu


In [5]:
!nvcc syncexample.cu -o syncexample

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!./syncexample

Input Array:  10 20 30 40 50 60 70 80 
Reversed:    80 70 60 50 40 30 20 10 


In [17]:
%%writefile addnum.cu
#include <iostream>
#include <cuda_runtime.h>


__global__ void addnums(int *d_c, int *d_a, int *d_b){
  *d_c = *d_a + *d_b;
}

int main(){
  int h_a = 10;
  int h_b = 20;
  int h_c;

  int *d_a, *d_b, *d_c;

  cudaMalloc((void**)&d_a, sizeof(int));
  cudaMalloc((void**)&d_b, sizeof(int));
  cudaMalloc((void**)&d_c, sizeof(int));

  cudaMemcpy(d_a, &h_a, sizeof(int), cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, &h_b, sizeof(int), cudaMemcpyHostToDevice);

  addnums<<<1,1>>>(d_c, d_a, d_b);

  cudaMemcpy(&h_c, d_c, sizeof(int), cudaMemcpyDeviceToHost);

  std::cout << h_a << " + " << h_b << " = " << h_c << std::endl;

  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_c);

  return 0;
}

Overwriting addnum.cu


In [18]:
!nvcc addnum.cu -o addnum

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [19]:
!./addnum

10 + 20 = 30


In [22]:
%%writefile block_cond.cu

#include <iostream>
#include <cuda_runtime.h>

#define ARR_SIZE 16
#define THREADS_PER_BLOCK 8

__global__ void condkernel(int *d_a){
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx<ARR_SIZE){
    if (idx%2==0) d_a[idx] = 0;
    else d_a[idx] = 1;
  }

  if (threadIdx.x == 0){
    d_a[idx] = 100;
  }
}

int main(){
  int h_a[ARR_SIZE];
  int *d_a;

  cudaMalloc((void**)&d_a, ARR_SIZE*sizeof(int));

  int numBlocks = ARR_SIZE/THREADS_PER_BLOCK;

  condkernel<<<numBlocks, THREADS_PER_BLOCK>>>(d_a);

  cudaMemcpy(&h_a, d_a, ARR_SIZE*sizeof(int), cudaMemcpyDeviceToHost);

  std::cout << "Result Array:\n";
    for (int i = 0; i < ARR_SIZE; i++) {
        std::cout << "Index " << i << ": " << h_a[i] << "\n";
    }

  cudaFree(d_a);
  return 0;

}

Overwriting block_cond.cu


In [23]:
!nvcc block_cond.cu -o block_cond

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [24]:
!./block_cond

Result Array:
Index 0: 100
Index 1: 1
Index 2: 0
Index 3: 1
Index 4: 0
Index 5: 1
Index 6: 0
Index 7: 1
Index 8: 100
Index 9: 1
Index 10: 0
Index 11: 1
Index 12: 0
Index 13: 1
Index 14: 0
Index 15: 1


In [33]:
%%writefile global_counter.cu

#include <iostream>
#include <cuda_runtime.h>

__device__ int global_block_count = 0;
__device__ int global_thread_count = 0;

__global__ void countKernel(int* d_arr, int N){
  int tid = blockIdx.x*blockDim.x + threadIdx.x;
  if (tid<N){
    d_arr[tid] = tid*2;

    atomicAdd(&global_thread_count, 1);
    __syncthreads();

    if (threadIdx.x==0) atomicAdd(&global_block_count, 1);
  }
}

int main(){
  int blocks = 4;
  int threads_per_block = 8;

  int total_threads = blocks * threads_per_block;

  int *d_arr;
  cudaMalloc((void**)&d_arr, total_threads*sizeof(int));

  countKernel<<<blocks, threads_per_block>>>(d_arr, total_threads);
  cudaDeviceSynchronize();

  int h_thread_count = 0;
  int h_block_count = 0;

  cudaMemcpyFromSymbol(&h_thread_count, global_thread_count, sizeof(int));
  cudaMemcpyFromSymbol(&h_block_count, global_block_count, sizeof(int));

  std::cout << "Total Threads Executed: " << h_thread_count << " (Expected: " << total_threads << ")\n";
  std::cout << "Total Blocks Completed: " << h_block_count  << " (Expected: " << blocks << ")\n";

  cudaFree(d_arr);
  return 0;
}

Overwriting global_counter.cu


In [31]:
!nvcc global_counter.cu -o global_counter

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [32]:
!./global_counter

Total Threads Executed: 32 (Expected: 32)
Total Blocks Completed: 4 (Expected: 4)


In [16]:
%%writefile sssp.cu

#include <iostream>
#include <climits>
#include <cuda_runtime.h>
#include <vector>

#define INF INT_MAX/2

__global__ void ssspKernel(int *row_ptr, int *col_ind, int *weights, int *dist, int *changed, int V){
  int u = blockIdx.x*blockDim.x + threadIdx.x;

  if (u>= V || dist[u] == INF) return;

  int start = row_ptr[u];
  int end = row_ptr[u+1];

  for (int i=start; i<end; i++){
    int v = col_ind[i];
    int w = weights[i];
    int new_dist = dist[u] + w;

    if (new_dist<dist[v]){
      int old_val = atomicMin(&dist[v], new_dist);
      if (new_dist<old_val){
        *changed = 1;
      }
    }
  }
}


int main(){
  int V = 5;
  int source = 0;

  std::vector<std::vector<std::pair<int, int>>>adj(V);
  adj[0] = {{1, 4}, {2, 2}};
  adj[1] = {{2, 1}, {3, 5}};
  adj[2] = {{3, 8}, {4, 10}};
  adj[3] = {{4, 2}};

  std::vector<int> h_row_ptr(V + 1, 0);
  std::vector<int> h_col_ind;
  std::vector<int> h_weights;

  for (int i=0; i<V; i++){
    h_row_ptr[i] = h_col_ind.size();
    for (auto &e : adj[i]){
      h_col_ind.push_back(e.first);
      h_weights.push_back(e.second);
    }
  }

  h_row_ptr[V] = h_col_ind.size();
  int E =  h_col_ind.size();  //total no. of edges


  std::vector<int> h_dist(V, INF);
  h_dist[source] = 0;

  int *d_row_ptr, *d_col_ind, *d_weights, *d_dist, *d_changed;

  cudaMalloc((void**)&d_row_ptr, (V+1)*sizeof(int));
  cudaMalloc((void**)&d_col_ind, E*sizeof(int));
  cudaMalloc((void**)&d_weights, E*sizeof(int));
  cudaMalloc((void**)&d_dist, V*sizeof(int));
  cudaMalloc((void**)&d_changed, sizeof(int));

  cudaMemcpy(d_row_ptr, h_row_ptr.data(), (V+1)*sizeof(int), cudaMemcpyHostToDevice);
  cudaMemcpy(d_col_ind, h_col_ind.data(), E*sizeof(int), cudaMemcpyHostToDevice);
  cudaMemcpy(d_weights, h_weights.data(), E*sizeof(int), cudaMemcpyHostToDevice);
  cudaMemcpy(d_dist, h_dist.data(), V*sizeof(int), cudaMemcpyHostToDevice);


  int threads_per_block = 256;
  int blocks = (V+threads_per_block-1)/threads_per_block;

  int h_changed = 1;

  while (h_changed){
    h_changed = 0;
    cudaMemcpy(d_changed, &h_changed, sizeof(int), cudaMemcpyHostToDevice);

    ssspKernel<<<blocks, threads_per_block>>>(d_row_ptr, d_col_ind, d_weights, d_dist, d_changed, V);

    cudaMemcpy(&h_changed, d_changed, sizeof(int), cudaMemcpyDeviceToHost);
    cudaDeviceSynchronize();
  }

  cudaMemcpy(h_dist.data(), d_dist, V*sizeof(int), cudaMemcpyDeviceToHost);

  std::cout << "Shortest distances from source " << source << ":\n";
  for (int i = 0; i < V; i++) {
      std::cout << "Node " << i << ": " << (h_dist[i] == INF ? -1 : h_dist[i]) << "\n";
  }

  cudaFree(d_row_ptr);
  cudaFree(d_col_ind);
  cudaFree(d_weights);
  cudaFree(d_dist);
  cudaFree(d_changed);

  return 0;

}

Overwriting sssp.cu


In [17]:
!nvcc sssp.cu -o sssp

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [18]:
!./sssp

Shortest distances from source 0:
Node 0: 0
Node 1: 4
Node 2: 2
Node 3: 9
Node 4: 11


In [9]:
%%writefile t2_sssp.cu

#include <iostream>
#include <vector>
#include <cuda_runtime.h>
#include <cmath>

#define INF 1e9

__global__ void sssp(int *row_ptr, int *col_ind, int *weights, int *dist, int *changed, int V){
	int u = blockIdx.x * blockDim.x + threadIdx.x;

	if (u>=V || dist[u] == INF) return;

	int start = row_ptr[u];
	int end = row_ptr[u+1];

	for (int i= start; i<end; i++){
		int v = col_ind[i];
		int w = weights[i];
		int new_dist = dist[u] + w;

		if (new_dist < dist[v]){
			int old_dist = atomicMin(&dist[v], new_dist);
			if (new_dist < old_dist) *changed = 1;
		}
	}
}

int main(){
	int V = 5;
	int source = 0;

	std::vector<std::vector<std::pair<int, int>>>adj(V);
	adj[0] = {{1, 4}, {2, 2}};
	adj[1] = {{2, 1}, {3, 5}};
	adj[2] = {{3, 8}, {4, 10}};
	adj[3] = {{4, 2}};

	std::vector<int> h_row_ptr(V+1, 0);
	std::vector<int> h_col_ind;
	std::vector<int> h_weights;

	for (int i=0; i<V; i++){
		h_row_ptr[i] = h_col_ind.size();
		for (auto &e : adj[i]){
			h_col_ind.push_back(e.first);
			h_weights.push_back(e.second);
		}
	}

	 h_row_ptr[V] = h_col_ind.size();
 	 int E =  h_col_ind.size();

	std::vector<int> h_dist(V, INF);
	h_dist[source] = 0;

	int *d_row_ptr, *d_col_ind, *d_weights, *d_dist, *d_changed;

  	cudaMalloc((void**)&d_row_ptr, (V+1)*sizeof(int));
	cudaMalloc((void**)&d_col_ind, E*sizeof(int));
	cudaMalloc((void**)&d_weights, E*sizeof(int));
	cudaMalloc((void**)&d_dist, V*sizeof(int));
	cudaMalloc((void**)&d_changed, sizeof(int));

	cudaMemcpy(d_row_ptr, h_row_ptr.data(), (V+1)*sizeof(int), cudaMemcpyHostToDevice);
 	cudaMemcpy(d_col_ind, h_col_ind.data(), E*sizeof(int), cudaMemcpyHostToDevice);
	cudaMemcpy(d_weights, h_weights.data(), E*sizeof(int), cudaMemcpyHostToDevice);
	cudaMemcpy(d_dist, h_dist.data(), V*sizeof(int), cudaMemcpyHostToDevice);


	int threads_per_block = 256;
	int blocks = (V+threads_per_block-1)/threads_per_block;

	int h_changed = 1;

	while (h_changed){
    		h_changed = 0;
    		cudaMemcpy(d_changed, &h_changed, sizeof(int), cudaMemcpyHostToDevice);

    		sssp<<<blocks, threads_per_block>>>(d_row_ptr, d_col_ind, d_weights, d_dist, d_changed, V);

    		cudaMemcpy(&h_changed, d_changed, sizeof(int), cudaMemcpyDeviceToHost);
    		cudaDeviceSynchronize();
  	}

  	cudaMemcpy(h_dist.data(), d_dist, V*sizeof(int), cudaMemcpyDeviceToHost);

  	std::cout << "Shortest distances from source " << source << ":\n";
  	for (int i = 0; i < V; i++) {
      		std::cout << "Node " << i << ": " << (h_dist[i] == INF ? -1 : h_dist[i]) << "\n";
  	}

  	cudaFree(d_row_ptr);
  	cudaFree(d_col_ind);
  	cudaFree(d_weights);
  	cudaFree(d_dist);
  	cudaFree(d_changed);

  return 0;
}


Overwriting t2_sssp.cu


In [10]:
!nvcc t2_sssp.cu -o t2_sssp

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [11]:
!./t2_sssp

Shortest distances from source 0:
Node 0: 0
Node 1: 4
Node 2: 2
Node 3: 9
Node 4: 11
